# CoatShield boundary finder: train on Kaggle

Pretrains the compact U-Net on OCT5k (real retinal OCT), fine-tunes it on synthetic pellet
scans, trains the same network from scratch for comparison, evaluates on the synthetic
validation set, exports ONNX (opset 17) and writes the manifest.

Before running:
1. On the laptop run `python scripts/package_for_kaggle.py`. It writes three zips to `dist/`.
2. Create three **private** Kaggle datasets from them, named exactly `coatshield-code`,
   `coatshield-synthetic` and `oct5k` (Kaggle unpacks the zips).
3. Attach all three to this notebook (Add input) and turn the GPU on (Settings > Accelerator).

The locked test set is never uploaded.

In [ ]:
import glob, os, shutil, subprocess, zipfile

def find(pattern):
    """First path under /kaggle/input matching the pattern (dataset folder names vary)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input - is the dataset attached?")
    return hits[0]

WORK = "/kaggle/working/coatshield"
if not os.path.exists(WORK):
    try:  # Kaggle usually unpacks an uploaded zip
        shutil.copytree(os.path.dirname(find("coatshield/pyproject.toml")), WORK)
    except FileNotFoundError:
        zipfile.ZipFile(find("coatshield_code.zip")).extractall("/kaggle/working")
os.chdir(WORK)
# torch and torchvision are preinstalled on Kaggle; install the rest.
subprocess.run("grep -v -E '^(torch|torchvision|--extra-index-url)' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt", shell=True, check=True)

links = {
    "data/raw/oct5k/OCT5k/Images": os.path.dirname(find("Images_Manual")),
    "data/raw/oct5k/OCT5k/Masks": os.path.dirname(find("Masks_Manual")),
    "data/synthetic/train": os.path.dirname(find("train/images.npy")),
    "data/synthetic/val": os.path.dirname(find("val/images.npy")),
}
for link, target in links.items():
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(target, link)
    print(link, "->", target)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn the accelerator on")

## 1. Pretrain on OCT5k (split by patient volume)

In [ ]:
import json
from coatshield.config import load_config
from coatshield.seg.train import run_stage

cfg = load_config()
device = "cuda" if torch.cuda.is_available() else "cpu"
pre = run_stage("pretrain", cfg, device, workers=2)
print(json.dumps({k: v for k, v in pre.items() if k != "history"}, indent=1))

## 2. Fine-tune on synthetic pellets, and 3. the same network from scratch

In [ ]:
fine = run_stage("finetune", cfg, device, workers=2)
scratch = run_stage("scratch", cfg, device, workers=2)
for name, info in (("fine-tuned", fine), ("from scratch", scratch)):
    print(name, info["val_boundaries"], "parameters", info["parameters"])

## 4. Evaluate, export ONNX and write the manifest

In [ ]:
subprocess.run(["python", "scripts/evaluate_seg.py"], check=True)
subprocess.run(["python", "scripts/export_onnx.py", "--checkpoint", "unet_pellets", "--name", "unet"], check=True)
print(open("models/manifest.json").read())

## 5. Collect the outputs

Download `coatshield_models.zip` and unpack it at the repository root: it holds `models/unet.onnx`, `models/manifest.json`, the checkpoints and the evaluation report.

In [ ]:
subprocess.run("zip -q -r /kaggle/working/coatshield_models.zip models reports/seg_*", shell=True, check=True)
print(os.path.getsize("/kaggle/working/coatshield_models.zip") / 1e6, "MB")